## 0. Task

Summarize a judgment. Official metric is BERTScore. Gold summaries overlap the source (extractive-ish) and are much shorter than the judgment.


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
SEARCH = [HERE, HERE.parent, Path("/kaggle/working")]
for base in (HERE, HERE.parent, HERE.parent.parent, HERE.parent.parent.parent):
    SEARCH.append(base)
    for phase in range(1, 8):
        SEARCH.append(base / f"phase{phase}")
        SEARCH.append(base / f"phase{phase}" / "code")
        SEARCH.append(base / "code" / f"phase{phase}")
for folder in SEARCH:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
print("python path heads:")
for folder in sys.path[:8]:
    print(" ", folder)


## 1. Abstractive path (required PEFT, not selected)

Causal models: `document + BOS + summary`, loss only on summary tokens.

Encoder–decoder: encode the judgment, decode the summary. **Do not** call `EncoderDecoderLM.forward` — that split was for Phase-1 LM.


In [ ]:
print("abstractive scripts: phase6_summ.py --arch {decoder,hybrid,efficient,encdec} --method lorma")
print("fluent but free to invent a holding")


## 2. Selected: extractive sentence rank, 720-word budget

Split the source into sentences, embed with the encoder–decoder, score by similarity to the document vector + position (head/tail) + lexical cues (`held`, `dismissed`, …). Greedy add if the word budget still fits. Force a few head and tail sentences so the holding is not dropped.


In [ ]:
import inspect
from phase6_summ import fill_extract

print(inspect.getsource(fill_extract))


## 3. Length control

The 720-word budget **is** the length control. No coverage penalty: extractive copy already stays faithful.


In [ ]:
print("command: python phase6_summ.py --arch encdec --extract --word-budget 720")
print("keep the encdec_extract_720 zip; do not overwrite it with abstractive SFT")


## 4. Zip


In [ ]:
print("prediction.json: {id: {text: summary}}")
